In [3]:
# Import libraries
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.config import CLEANED_DATA_PATH, FEATURED_DATA_PATH
from src.features import create_churn_features

In [4]:
# load cleaned data
df = pd.read_csv(CLEANED_DATA_PATH)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnFlag
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [5]:
# apply feature engineering
df_feat = create_churn_features(df)
df_feat.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Churn,ChurnFlag,tenure_group,avg_monthly_value,num_services,has_protection,contract_risk,is_electronic_check,is_new_customer,is_high_value
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,0,0-12,14.925000,1,0,2,1,1,0
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,No,0,24-48,53.985714,3,1,1,0,0,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,Yes,1,0-12,36.050000,3,1,2,0,1,0
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,No,0,24-48,40.016304,3,1,1,0,0,0
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,Yes,1,0-12,50.550000,1,0,2,1,1,1


In [10]:
# inspect new columns
new_cols = set(df_feat.columns) - set(df.columns)
list(new_cols)

['is_electronic_check',
 'contract_risk',
 'is_new_customer',
 'avg_monthly_value',
 'is_high_value',
 'tenure_group',
 'has_protection',
 'num_services']

In [13]:
# check distributions
df_feat[list(new_cols)].describe(include="all")

,is_electronic_check,contract_risk,is_new_customer,avg_monthly_value,is_high_value,tenure_group,has_protection,num_services
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7032,7043.000000,7043.000000
unique,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,0-12,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,2175,NaN,NaN
mean,0.335794,1.309527,0.310379,58.990789,0.499077,NaN,0.536561,3.362914
std,0.472301,0.833755,0.462682,30.579745,0.500035,NaN,0.498697,2.062031
min,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000
25%,0.000000,1.000000,0.000000,26.041493,0.000000,NaN,0.000000,1.000000
50%,0.000000,2.000000,0.000000,60.937879,0.000000,NaN,1.000000,3.000000
75%,1.000000,2.000000,1.000000,84.830742,1.000000,NaN,1.000000,5.000000


In [15]:
# Validate features - Tenure Group vs Churn
pd.crosstab(df_feat["tenure_group"], df_feat["Churn"], normalize="index") * 100

Churn,No,Yes
tenure_group,,
0-12,52.321839,47.678161
12-24,71.289062,28.710938
24-48,79.611041,20.388959
48-60,85.576923,14.423077
60+,93.390192,6.609808


In [16]:
pd.crosstab(df_feat["num_services"], df_feat["Churn"], normalize="index") * 100

Churn,No,Yes
num_services,,
0,56.250000,43.750000
1,78.894768,21.105232
2,67.171717,32.828283
3,63.523316,36.476684
4,68.655098,31.344902
5,74.449339,25.550661
6,77.514793,22.485207
7,87.594937,12.405063
8,94.711538,5.288462


In [17]:
df_feat.to_csv(FEATURED_DATA_PATH, index=False)
print("Feature dataset saved!")

Feature dataset saved!


## Feature Engineering Summary

The following features were created to improve predictive performance and capture business-relevant signals:

- tenure_group: groups customers by lifecycle stage
- avg_monthly_value: estimated value per customer
- num_services: total services subscribed
- has_protection: whether customer has security/support services
- contract_risk: risk level based on contract type
- is_electronic_check: indicator of high-risk payment method
- is_new_customer: early-stage customer flag
- is_high_value: above-median revenue customer

These features are designed to reflect real-world churn drivers and improve model interpretability.